In [ ]:
History features (computed without peeking forward)

In [1]:
import pandas as pd, numpy as np

panel = (pd.read_parquet("../data/panel.parquet")
         .sort_values(["StockCode", "Week"]).reset_index(drop=True))

g = panel.groupby("StockCode")
panel["lag1_log_units"]  = g["log_units"].shift(1)
panel["roll4_log_units"] = g["log_units"].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
panel["lag1_log_price"]  = g["log_price"].shift(1)
panel["price_change"]    = panel["log_price"] - panel["lag1_log_price"]
panel["weeks_gap"]       = g["Week"].diff().dt.days / 7
panel["cat_code"]        = panel["category"].astype("category").cat.codes

model_df = panel.dropna(subset=["lag1_log_units", "lag1_log_price"]).copy()
print(model_df.shape)

(64994, 25)


Time-based split

In [2]:
weeks = np.sort(model_df["Week"].unique())
cutoff = weeks[int(len(weeks) * 0.8)]
print("Test starts:", pd.Timestamp(cutoff).date())

train = model_df[model_df["Week"] < cutoff]
test  = model_df[model_df["Week"] >= cutoff]
print(train.shape, test.shape)

Test starts: 2011-07-18
(52447, 25) (12547, 25)


Baselines and model


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

feats = ["log_price", "price_change", "lag1_log_units", "roll4_log_units",
         "weeks_gap", "week_of_year", "month", "is_q4", "cat_code"]

y_tr, y_te = train["log_units"], test["log_units"]

# Baselines
naive     = test["lag1_log_units"]
prod_mean = test["StockCode"].map(train.groupby("StockCode")["log_units"].mean()).fillna(y_tr.mean())

# XGBoost
model = xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                         subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(train[feats], y_tr)
pred = model.predict(test[feats])

def report(name, p):
    mae  = mean_absolute_error(y_te, p)
    rmse = np.sqrt(mean_squared_error(y_te, p))
    wape = np.abs(test["units"] - np.exp(p)).sum() / test["units"].sum()
    print(f"{name:<22} MAE(log)={mae:.3f}  RMSE(log)={rmse:.3f}  WAPE(units)={wape:.1%}")

report("Last observed week", naive)
report("Product train mean", prod_mean)
report("XGBoost", pred)

Last observed week     MAE(log)=1.100  RMSE(log)=1.461  WAPE(units)=93.7%
Product train mean     MAE(log)=1.012  RMSE(log)=1.268  WAPE(units)=79.8%
XGBoost                MAE(log)=0.806  RMSE(log)=1.023  WAPE(units)=68.8%


What the model learned

In [4]:
imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.round(3))

log_price          0.263
roll4_log_units    0.226
price_change       0.213
lag1_log_units     0.151
cat_code           0.035
week_of_year       0.033
month              0.031
is_q4              0.026
weeks_gap          0.023
dtype: float32


does the model agree with the regression?

In [5]:
def simulate(delta):
    t = test.copy()
    shift = np.log(1 + delta)
    t["log_price"] = t["log_price"] + shift
    t["price_change"] = t["price_change"] + shift   # keep the two price features consistent
    return model.predict(t[feats])

base = model.predict(test[feats])

for d in [-0.10, -0.05, 0.05, 0.10]:
    p = simulate(d)
    implied = (p - base).mean() / np.log(1 + d)
    print(f"Price {d:+.0%}: implied elasticity = {implied:.2f}")

Price -10%: implied elasticity = -4.36
Price -5%: implied elasticity = -5.24
Price +5%: implied elasticity = -2.81
Price +10%: implied elasticity = -1.62


In [ ]:
out = test[["StockCode", "Week", "category", "units", "price"]].copy()
out["pred_units"] = np.exp(pred)
out.to_csv("../data/test_predictions.csv", index=False)